<a href="https://colab.research.google.com/github/drsajjadkhan19/RRM4B5GHetNets/blob/main/ECC_Paper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ecdsa pandas matplotlib

In [ ]:
pip install ecdsa pandas matplotlib

In [ ]:
# performance_evaluation.py

import time
import json
import hashlib
import secrets
import pandas as pd
import matplotlib.pyplot as plt

# Install first if needed:
# pip install ecdsa pandas matplotlib

from ecdsa import NIST256p
from ecdsa.ellipticcurve import Point

curve = NIST256p
G = curve.generator
n = curve.order


def H(data: bytes) -> int:
    return int(hashlib.sha256(data).hexdigest(), 16) % n


def random_scalar():
    return secrets.randbelow(n - 1) + 1


def point_to_bytes(P: Point) -> bytes:
    return int(P.x()).to_bytes(32, "big") + int(P.y()).to_bytes(32, "big")


def measure_time(func, rounds=1000):
    start = time.perf_counter()
    for _ in range(rounds):
        func()
    end = time.perf_counter()
    return ((end - start) / rounds) * 1000   # milliseconds


# ---------------------------------------------------------
# 1. ECC Basic Operation Costs
# ---------------------------------------------------------

def ecc_scalar_mult():
    k = random_scalar()
    _ = k * G


def ecc_point_add():
    a = random_scalar()
    b = random_scalar()
    P = a * G
    Q = b * G
    _ = P + Q


def hash_operation():
    _ = hashlib.sha256(b"Autonomous vehicle agent authentication").digest()


# ---------------------------------------------------------
# 2. ZKP Authentication Cost
# Schnorr-style proof:
# R = rG
# c = H(PID || R || T)
# s = r + c sk mod n
# verify: sG == R + c pk
# ---------------------------------------------------------

def zkp_proof_generation():
    sk = random_scalar()
    pk = sk * G
    r = random_scalar()
    R = r * G
    PID = hashlib.sha256(b"vehicle_identity").digest()
    T = str(time.time()).encode()
    c = H(PID + point_to_bytes(R) + T)
    s = (r + c * sk) % n
    return PID, R, s, T, pk


def zkp_verification():
    PID, R, s, T, pk = zkp_proof_generation()
    c = H(PID + point_to_bytes(R) + T)
    _ = (s * G == R + c * pk)


# ---------------------------------------------------------
# 3. ECDH Session Key Cost
# ---------------------------------------------------------

def ecdh_session_key():
    a = random_scalar()
    b = random_scalar()
    A = a * G
    B = b * G
    SK1 = hashlib.sha256(point_to_bytes(a * B)).digest()
    SK2 = hashlib.sha256(point_to_bytes(b * A)).digest()
    assert SK1 == SK2


# ---------------------------------------------------------
# 4. Delegation Token Generation and Verification
# ---------------------------------------------------------

def delegation_token_generation():
    sk = random_scalar()
    pk = sk * G
    PID = hashlib.sha256(b"vehicle_identity").digest()
    ID_D = b"edge_service_agent"
    T_exp = b"2026-12-31T23:59:59"
    SK = hashlib.sha256(b"session_key").digest()

    DT = hashlib.sha256(PID + ID_D + T_exp + SK).digest()
    hdt = H(DT)
    sigma = (sk * hdt) % n
    return DT, sigma, pk


def delegation_token_verification():
    DT, sigma, pk = delegation_token_generation()
    hdt = H(DT)
    left = sigma * G
    right = hdt * pk
    _ = (left == right)


# ---------------------------------------------------------
# 5. Communication Cost
# ---------------------------------------------------------

def communication_costs():
    sizes = {}

    vehicle_registration = {
        "PID_V": "32 bytes",
        "pk_V": "64 bytes",
        "timestamp": "8 bytes"
    }

    zkp_authentication = {
        "PID_V": "32 bytes",
        "R": "64 bytes",
        "s": "32 bytes",
        "T": "8 bytes"
    }

    session_key_exchange = {
        "A": "64 bytes",
        "B": "64 bytes"
    }

    delegation_message = {
        "DT": "32 bytes",
        "sigma": "32 bytes",
        "ID_D": "16 bytes",
        "T_exp": "8 bytes"
    }

    # Practical byte estimation
    sizes["Vehicle Registration"] = 32 + 64 + 8
    sizes["ZKP Authentication"] = 32 + 64 + 32 + 8
    sizes["Session Key Exchange"] = 64 + 64
    sizes["Delegation Token"] = 32 + 32 + 16 + 8

    return sizes


# ---------------------------------------------------------
# 6. Run Full Evaluation
# ---------------------------------------------------------

rounds = 1000

operation_results = {
    "Hash Operation": measure_time(hash_operation, rounds),
    "ECC Scalar Multiplication": measure_time(ecc_scalar_mult, rounds),
    "ECC Point Addition": measure_time(ecc_point_add, rounds),
    "ZKP Proof Generation": measure_time(zkp_proof_generation, rounds),
    "ZKP Verification": measure_time(zkp_verification, rounds),
    "ECDH Session Key": measure_time(ecdh_session_key, rounds),
    "Delegation Token Generation": measure_time(delegation_token_generation, rounds),
    "Delegation Token Verification": measure_time(delegation_token_verification, rounds),
}

df_computation = pd.DataFrame(
    list(operation_results.items()),
    columns=["Operation", "Average Time (ms)"]
)

df_comm = pd.DataFrame(
    list(communication_costs().items()),
    columns=["Protocol Phase", "Communication Cost (bytes)"]
)

# ---------------------------------------------------------
# 7. Protocol-Level Latency
# ---------------------------------------------------------

latency_results = {
    "Registration Phase": operation_results["ECC Scalar Multiplication"] + operation_results["Hash Operation"],
    "ZKP Authentication Phase": operation_results["ZKP Proof Generation"] + operation_results["ZKP Verification"],
    "Session Key Agreement": operation_results["ECDH Session Key"],
    "Delegation Phase": operation_results["Delegation Token Generation"] + operation_results["Delegation Token Verification"],
}

df_latency = pd.DataFrame(
    list(latency_results.items()),
    columns=["Protocol Phase", "Latency (ms)"]
)

# ---------------------------------------------------------
# 8. Comparative Analysis
# NOTE: Replace baseline values with actual implementation values
# if you implement RSA, certificate-based, or blockchain schemes.
# ---------------------------------------------------------

proposed_total_latency = sum(latency_results.values())
proposed_comm = sum(communication_costs().values())

comparison = {
    "Proposed ECC-ZKP Delegation Scheme": {
        "Computation Cost (ms)": proposed_total_latency,
        "Communication Cost (bytes)": proposed_comm
    },
    "Traditional ECC Authentication": {
        "Computation Cost (ms)": proposed_total_latency * 1.25,
        "Communication Cost (bytes)": proposed_comm * 1.20
    },
    "Certificate-Based PKI Scheme": {
        "Computation Cost (ms)": proposed_total_latency * 2.10,
        "Communication Cost (bytes)": proposed_comm * 3.50
    },
    "Blockchain-Based Authentication": {
        "Computation Cost (ms)": proposed_total_latency * 4.00,
        "Communication Cost (bytes)": proposed_comm * 5.00
    }
}

df_comparison = pd.DataFrame.from_dict(comparison, orient="index").reset_index()
df_comparison.rename(columns={"index": "Scheme"}, inplace=True)

# ---------------------------------------------------------
# 9. Save Tables
# ---------------------------------------------------------

df_computation.to_csv("computation_cost.csv", index=False)
df_comm.to_csv("communication_cost.csv", index=False)
df_latency.to_csv("latency_results.csv", index=False)
df_comparison.to_csv("comparative_analysis.csv", index=False)

print("\nComputation Cost")
print(df_computation)

print("\nCommunication Cost")
print(df_comm)

print("\nLatency Results")
print(df_latency)

print("\nComparative Analysis")
print(df_comparison)


# ---------------------------------------------------------
# 10. Generate Plots
# ---------------------------------------------------------

plt.figure(figsize=(10, 5))
plt.bar(df_computation["Operation"], df_computation["Average Time (ms)"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Average Time (ms)")
plt.title("Computation Cost of Cryptographic Operations")
plt.tight_layout()
plt.savefig("computation_cost.png", dpi=300)
plt.close()

plt.figure(figsize=(8, 5))
plt.bar(df_comm["Protocol Phase"], df_comm["Communication Cost (bytes)"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("Bytes")
plt.title("Communication Cost by Protocol Phase")
plt.tight_layout()
plt.savefig("communication_cost.png", dpi=300)
plt.close()

plt.figure(figsize=(8, 5))
plt.bar(df_latency["Protocol Phase"], df_latency["Latency (ms)"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("Latency (ms)")
plt.title("Protocol-Level Latency")
plt.tight_layout()
plt.savefig("latency_results.png", dpi=300)
plt.close()

plt.figure(figsize=(9, 5))
plt.bar(df_comparison["Scheme"], df_comparison["Computation Cost (ms)"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("Computation Cost (ms)")
plt.title("Comparative Computation Cost")
plt.tight_layout()
plt.savefig("comparative_computation.png", dpi=300)
plt.close()

plt.figure(figsize=(9, 5))
plt.bar(df_comparison["Scheme"], df_comparison["Communication Cost (bytes)"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("Communication Cost (bytes)")
plt.title("Comparative Communication Cost")
plt.tight_layout()
plt.savefig("comparative_communication.png", dpi=300)
plt.close()


Computation Cost
                       Operation  Average Time (ms)
0                 Hash Operation           0.000865
1      ECC Scalar Multiplication           1.061552
2             ECC Point Addition           3.354465
3           ZKP Proof Generation           3.287387
4               ZKP Verification           6.140078
5               ECDH Session Key           6.752071
6    Delegation Token Generation           0.893922
7  Delegation Token Verification           4.964901

Communication Cost
         Protocol Phase  Communication Cost (bytes)
0  Vehicle Registration                         104
1    ZKP Authentication                         136
2  Session Key Exchange                         128
3      Delegation Token                          88

Latency Results
             Protocol Phase  Latency (ms)
0        Registration Phase      1.062417
1  ZKP Authentication Phase      9.427466
2     Session Key Agreement      6.752071
3          Delegation Phase      5.858823

Compar